# 코로나19 초기 서울시 야간광량·NO₂ 변화와 도시 기능별 연관성 분석

**자료**: 자치구별 2019년·2020년 3~5월 평균 야간광량(Suomi NPP VIIRS), NO₂ 대류권 기둥농도(Sentinel-5P TROPOMI), 토지 용도별 대표 집단(입지계수 LQ로 선정)

| 단계 | 세부목적 | 셀 |
|---|---|---|
| 준비 | 링크 입력, 라이브러리, 자료 불러오기 | 1~4 |
| 세부목적 (1) | 25개 자치구별 전년 동기 대비 변화율(%) 산출, 변화 방향·크기·공간적 분포 파악 | 5 |
| 세부목적 (2) | 25개 자치구 전체: 야간광량 변화율과 NO₂ 변화율의 회귀분석·상관관계 정량화 | 6 |
| 세부목적 (3) | LQ로 선정한 상업·주거·공업·녹지 특화형 집단별 회귀분석·상관관계 → 전체와 비교 | 7~8 |
| 저장 | 결과 엑셀과 그림 저장 | 9 |

**사용법**: 1번 셀의 빈칸 두 곳에 CSV 파일 링크를 붙여 넣고, 메뉴에서 **런타임 → 모두 실행**을 누르세요.

## 1. 설정: 파일 링크 입력
링크는 다음 형식을 모두 쓸 수 있습니다.
- 구글 드라이브 공유 링크 (공유 설정: **링크가 있는 모든 사용자**)
- 구글 스프레드시트 링크
- GitHub 파일 링크, 일반 웹 주소(https://…csv)
- 코랩에 마운트한 드라이브 경로 (`/content/drive/MyDrive/…csv`)

In [ ]:
# ▶ [1단계] 설정: 파일 링크, 토지 용도별 집단, 분석 옵션을 정하는 칸 (이 셀만 고치면 됩니다)

# ===== ① 파일 링크를 따옴표 안에 붙여 넣으세요 (월별 자료면 3~5월 평균을 자동 계산) =====
NTL_CSV_LINK    = ''   # 자치구별 2019·2020년 3~5월 평균 야간광량
NO2_CSV_LINK    = ''   # 자치구별 2019·2020년 3~5월 평균 NO2 기둥농도

# ===== ② 토지 용도별 대표 집단 (입지계수 LQ로 선정한 결과) =====
GROUPS = {
    '상업 특화형': ['종로구', '중구', '용산구', '강남구', '송파구'],
    '주거 특화형': ['광진구', '동대문구', '중랑구', '성북구', '서대문구', '마포구', '양천구', '동작구'],
    '공업 특화형': ['성동구', '도봉구', '강서구', '구로구', '금천구', '영등포구'],
    '녹지 특화형': ['강북구', '노원구', '은평구', '관악구', '서초구', '강동구'],
}

# ===== ③ 열 이름 (비워 두면 자동으로 찾습니다) =====
GU_COL   = None   # 자치구 이름 열. 예: '자치구'
COL_2019 = None   # 2019년 값 열.   예: '2019'
COL_2020 = None   # 2020년 값 열.   예: '2020'

# ===== ④ 분석 설정 =====
# 변화율(%) = (2020년 3~5월 평균 − 2019년 3~5월 평균) ÷ 2019년 3~5월 평균 × 100   (음수 = 감소)
# 보조 분석으로 변화량(2019 − 2020)을 보고 싶으면 'diff'로 바꾸세요.
CHANGE_DEF   = 'pct'
ALPHA        = 0.05
MONTHS       = (3, 4, 5)     # 월별 자료일 때 평균할 달
SAVE_TO_DRIVE = False        # True로 바꾸면 결과를 구글 드라이브 OUT_DIR에도 저장
OUT_DIR = '/content/drive/MyDrive/ntl_no2_results'

## 2. 라이브러리와 한글 폰트

In [ ]:
# ▶ [2단계] 준비: 라이브러리 불러오기, 한글 폰트 설정, 결과 저장 폴더 만들기

# --- 2-1. 기본 라이브러리
import os, re, io, json, warnings, subprocess, urllib.request
warnings.filterwarnings('ignore')

# --- 2-2. 코랩에서 실행 중인지 확인하고, 코랩이면 한글 폰트(나눔고딕) 설치
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
except ImportError:
    IN_COLAB = False

# --- 2-3. 지도(geopandas), 표(pandas), 그래프(matplotlib), 통계(scipy·statsmodels) 라이브러리
try:
    import geopandas as gpd
except ImportError:                      # 코랩에 없을 때만 설치
    subprocess.run('pip -q install geopandas', shell=True)
    import geopandas as gpd
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from scipy import stats
import statsmodels.api as sm

# --- 2-4. 그래프에 한글이 깨지지 않도록 설치된 한글 폰트를 등록
for path in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
             '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']:
    if os.path.exists(path):
        fm.fontManager.addfont(path)
        plt.rc('font', family=fm.FontProperties(fname=path).get_name())
        break
# 음수 기호(−)가 네모로 깨지지 않게 하고, 표의 숫자를 유효숫자 4자리로 보기 좋게 표시
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', lambda v: f'{v:,.4g}')

# --- 2-5. 결과(그림·엑셀)를 저장할 폴더 만들기, 필요하면 구글 드라이브 연결
LOCAL_OUT = '/content/results' if IN_COLAB else os.path.abspath('results')
os.makedirs(LOCAL_OUT, exist_ok=True)
if IN_COLAB and (SAVE_TO_DRIVE or any(str(l).startswith('/content/drive') for l in [NTL_CSV_LINK, NO2_CSV_LINK])):
    from google.colab import drive
    drive.mount('/content/drive')
if SAVE_TO_DRIVE:
    os.makedirs(OUT_DIR, exist_ok=True)
print('준비 완료 / 결과 폴더:', LOCAL_OUT)

## 3. 자료 불러오기
- 링크 종류를 알아서 판별하고, 인코딩(UTF-8, CP949)도 자동으로 맞춥니다.
- 자치구 이름 열과 2019·2020년 값 열을 자동으로 찾습니다. 못 찾으면 1번 셀의 `GU_COL`, `COL_2019`, `COL_2020`을 직접 적으세요.
- 연도가 행으로 길게 정리된 표(자치구, 연도, 값)도 처리합니다.

In [ ]:
# ▶ [3단계] 자료 불러오기: CSV 링크 → 표 읽기 → (자치구, 2019년 값, 2020년 값) 형태로 정리 → 품질 점검

# --- 3-1. 서울시 25개 자치구 이름 목록 (자료 점검의 기준)
GU_NAMES = ['종로구', '중구', '용산구', '성동구', '광진구', '동대문구', '중랑구', '성북구', '강북구', '도봉구',
            '노원구', '은평구', '서대문구', '마포구', '양천구', '강서구', '구로구', '금천구', '영등포구', '동작구',
            '관악구', '서초구', '강남구', '송파구', '강동구']

# --- 3-2. 링크 변환: 공유 링크를 파일을 바로 내려받을 수 있는 주소로 바꿈
def to_direct_url(link):
    """구글 드라이브·스프레드시트·GitHub 공유 링크를 바로 내려받을 수 있는 주소로 바꾼다."""
    link = link.strip()
    m = re.search(r'docs\.google\.com/spreadsheets/d/([\w-]+)', link)
    if m:
        gid = re.search(r'gid=(\d+)', link)
        return f'https://docs.google.com/spreadsheets/d/{m.group(1)}/export?format=csv' + (f'&gid={gid.group(1)}' if gid else '')
    m = re.search(r'drive\.google\.com/(?:file/d/|open\?id=|uc\?(?:export=download&)?id=)([\w-]+)', link)
    if m:
        return f'https://drive.google.com/uc?export=download&id={m.group(1)}'
    m = re.match(r'https://github\.com/([^/]+/[^/]+)/blob/(.+)', link)
    if m:
        return f'https://raw.githubusercontent.com/{m.group(1)}/{m.group(2)}'
    return link

# --- 3-3. CSV 읽기: 링크에서 파일을 받아 표로 읽음 (한글 인코딩 UTF-8·CP949 자동 판별)
def read_csv_link(link, label):
    assert link, f'1번 셀에 {label} 링크를 입력하세요.'
    src = to_direct_url(link)
    raw = open(src, 'rb').read() if os.path.exists(src) else urllib.request.urlopen(src).read()
    if raw[:15].lower().startswith(b'<!doctype html') or raw[:6].lower() == b'<html>':
        raise ValueError(f'{label}: 파일 대신 웹페이지가 내려왔습니다. 공유 설정을 "링크가 있는 모든 사용자"로 바꿨는지 확인하세요.')
    for enc in ('utf-8-sig', 'cp949', 'euc-kr'):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'{label}: 인코딩을 읽을 수 없습니다.')

# --- 3-4. 자치구 이름 정리: "서울특별시 강남구" → "강남구"
def clean_gu(s):
    s = str(s).strip()
    m = re.search(r'(\S+구)$', s)
    return m.group(1) if m else s

# --- 3-5. 표에서 자치구 이름이 들어 있는 열을 자동으로 찾음
def find_gu_col(df):
    if GU_COL:
        return GU_COL
    for c in df.columns:
        if df[c].astype(str).map(clean_gu).isin(GU_NAMES).sum() >= 20:
            return c
    raise ValueError(f'자치구 이름 열을 찾지 못했습니다. 열 목록: {list(df.columns)} → GU_COL을 지정하세요.')

# --- 3-6. 표 정리: 어떤 모양의 CSV든 (자치구, y2019, y2020) 세 열로 맞춤
def tidy_two_years(df, label):
    """어떤 모양의 표든 (자치구, 2019, 2020) 세 열로 정리한다."""
    df = df.copy(); df.columns = [str(c).strip() for c in df.columns]
    gcol = find_gu_col(df)
    df['자치구'] = df[gcol].astype(str).map(clean_gu)
    df = df[df['자치구'].isin(GU_NAMES)]
    # (가로 형식) 2019·2020이 들어간 열 이름을 찾음. 3개월 평균 열이 있으면 그 열을 우선 사용
    def pick_year_col(y):
        cands = [c for c in df.columns if str(y) in c and not re.search('비율|변화|ratio|chg|change', c, re.I)]
        pref = [c for c in cands if re.search('평균|mean|avg', c, re.I)]
        if pref:
            return pref[0]
        if len(cands) > 1:   # 예: 2019_3월, 2019_4월, 2019_5월 → 여러 달 열을 평균
            col = f'{y}년 {len(cands)}개 열 평균'
            df[col] = df[cands].apply(lambda s: pd.to_numeric(s.astype(str).str.replace(',', ''), errors='coerce')).mean(axis=1)
            print(f'{label}: {y}년 열이 여러 개라 평균함 → {cands}')
            return col
        return (cands or [None])[0]
    if COL_2019 and COL_2020:
        c19, c20 = COL_2019, COL_2020
    else:
        c19, c20 = pick_year_col(2019), pick_year_col(2020)
    if c19 and c20:
        out = df[['자치구', c19, c20]].rename(columns={c19: 'y2019', c20: 'y2020'})
    else:  # 긴 형식: 자치구 · 연도 · (월) · 값
        # (세로 형식) 자치구·연도·(월)·값 열을 찾아서, 월별 자료면 3~5월 평균을 계산
        num = lambda c: pd.to_numeric(df[c], errors='coerce')
        is_num = lambda c: num(c).notna().mean() > 0.9
        others = [c for c in df.columns if c not in (gcol, '자치구')]
        ycol = next((c for c in others if is_num(c) and num(c).isin([2019, 2020]).all()), None)
        mcol = next((c for c in others if c != ycol and is_num(c) and num(c).between(1, 12).all()
                     and num(c).nunique() > 1 and re.search('month|월', c, re.I)), None)
        cands = [c for c in others if c not in (ycol, mcol) and is_num(c)]
        pref = [c for c in cands if re.search('NO2|NTL|mean|평균|값|value|rad', c, re.I)]
        vcol = (pref or cands or [None])[0]
        if not (ycol and vcol):
            raise ValueError(f'{label}: 2019·2020년 값 열을 찾지 못했습니다. 열 목록: {list(df.columns)} → COL_2019, COL_2020을 지정하세요.')
        df[vcol] = num(vcol)
        if mcol:   # 월별 자료 → 분석 월(3~5월)만 골라 자치구·연도별 평균
            df = df[num(mcol).isin(MONTHS)]
            cnt = df.groupby(['자치구', num(ycol)])[mcol].nunique()
            print(f'{label}: 월별 자료("{mcol}" 열) → {MONTHS}월 평균으로 계산 (자치구·연도별 월 수: {cnt.min()}~{cnt.max()}개)')
        out = df.pivot_table(index='자치구', columns=num(ycol), values=vcol, aggfunc='mean').reset_index()
        out.columns = ['자치구', 'y2019', 'y2020']
        print(f'{label}: 값 열 = "{vcol}", 연도 열 = "{ycol}"')
    # 숫자로 변환하고 25개 자치구가 모두 있는지 확인
    out[['y2019', 'y2020']] = out[['y2019', 'y2020']].apply(lambda s: pd.to_numeric(s.astype(str).str.replace(',', ''), errors='coerce'))
    missing = set(GU_NAMES) - set(out['자치구'])
    assert not missing, f'{label}: 빠진 자치구가 있습니다 → {missing}'
    print(f'{label}: 2019년 열 = "{c19 or "(긴 형식)"}", 2020년 열 = "{c20 or "(긴 형식)"}", 25개 자치구 확인')
    return out.groupby('자치구', as_index=False).mean()

# --- 3-7. 실제 실행: 두 CSV를 읽고 정리
ntl_raw = read_csv_link(NTL_CSV_LINK, '야간광량 CSV')
no2_raw = read_csv_link(NO2_CSV_LINK, 'NO2 CSV')
ntl = tidy_two_years(ntl_raw, '야간광량')
no2 = tidy_two_years(no2_raw, 'NO2')

# --- 3-8. 자료 품질 점검
# 자료 품질 점검: 서로 다른 자치구가 완전히 같은 값을 갖는지 확인
for label, d in [('야간광량', ntl), ('NO2', no2)]:
    dup = d[d.duplicated(['y2019', 'y2020'], keep=False)]
    if len(dup):
        pairs = dup.groupby(['y2019', 'y2020'])['자치구'].apply(lambda s: ' = '.join(s)).tolist()
        print(f'⚠️ {label}: 값이 완전히 같은 자치구 {len(dup)}개 → {pairs}')
        print('   추출 해상도(scale)가 너무 커서 이웃 자치구가 같은 픽셀 값을 받았을 수 있습니다. 원자료 추출 설정을 확인하세요.')

# --- 3-9. 단위 정리
# NO2 단위: 값이 아주 작으면(mol/m²) 읽기 쉽게 µmol/m²로 바꾼다 (상관·회귀 결과에는 영향 없음)
NO2_UNIT = 'mol/m²'
if no2[['y2019', 'y2020']].abs().median().median() < 0.01:
    no2[['y2019', 'y2020']] *= 1e6
    NO2_UNIT = 'µmol/m²'
print('NO2 단위:', NO2_UNIT)
display(ntl.head(3)); display(no2.head(3))

## 4. 자치구 경계 (지도용)

In [ ]:
# ▶ [4단계] 지도용 자치구 경계 불러오기 (통계청 경계를 단순화한 공개 GeoJSON)

GU_URL = ('https://raw.githubusercontent.com/southkorea/seoul-maps/master/'
          'kostat/2013/json/seoul_municipalities_geo_simple.json')
# 경계 파일을 인터넷에서 받아 지도 데이터(GeoDataFrame)로 만들고, 자치구 이름 열만 남김
gu = gpd.GeoDataFrame.from_features(json.load(urllib.request.urlopen(GU_URL))['features'], crs='EPSG:4326')
gu = gu.rename(columns={'name': '자치구'})[['자치구', 'geometry']]
print(len(gu), '개 자치구 경계 불러옴')

## 5. 세부목적 (1): 자치구별 변화율(%) 산출과 공간적 분포
**변화율(%) = (2020년 3~5월 평균 − 2019년 3~5월 평균) ÷ 2019년 3~5월 평균 × 100** (음수 = 감소, 양수 = 증가)

평가 지표: 자치구별 변화율, 25개 자치구의 평균·중앙값·최솟값·최댓값·표준편차, 감소한 자치구 수, 변화율 지도

In [ ]:
# ▶ [5단계] 세부목적 (1): 자치구별 변화율 산출 → 분포 통계표 → 변화율 지도

# --- 5-1. 야간광량 표와 NO2 표를 자치구 기준으로 합침
data = ntl.merge(no2, on='자치구', suffixes=('_ntl', '_no2'))

# --- 5-2. 변화 계산 (1번 셀의 CHANGE_DEF에 따라)
if CHANGE_DEF == 'diff':
    data['ntl_chg'] = data['y2019_ntl'] - data['y2020_ntl']
    data['no2_chg'] = data['y2019_no2'] - data['y2020_no2']
    LBL_NTL, LBL_NO2 = '야간광량 변화량 (2019−2020)', f'NO$_2$ 변화량 (2019−2020, {NO2_UNIT})'
    TXT_NTL, TXT_NO2 = '야간광량 변화량(2019−2020)', f'NO₂ 변화량(2019−2020, {NO2_UNIT})'
    is_decrease = lambda s: s > 0
# 기본값 pct: 변화율(%) = (2020년 평균 − 2019년 평균) ÷ 2019년 평균 × 100 → 양수 = 증가, 음수 = 감소
else:
    data['ntl_chg'] = (data['y2020_ntl'] - data['y2019_ntl']) / data['y2019_ntl'] * 100
    data['no2_chg'] = (data['y2020_no2'] - data['y2019_no2']) / data['y2019_no2'] * 100
    LBL_NTL, LBL_NO2 = '야간광량 변화율(%)', 'NO$_2$ 변화율(%)'
    TXT_NTL, TXT_NO2 = '야간광량 변화율(%)', 'NO₂ 변화율(%)'
    is_decrease = lambda s: s < 0

# --- 5-3. 세부목적 (1) 평가 지표 표
# 평가 지표: 평균, 중앙값, 최솟값, 최댓값, 표준편차 + 감소한 자치구 수
summary1 = pd.DataFrame({
    TXT_NTL: data['ntl_chg'].agg(['mean', 'median', 'min', 'max', 'std']),
    TXT_NO2: data['no2_chg'].agg(['mean', 'median', 'min', 'max', 'std'])}).T
summary1.columns = ['평균', '중앙값', '최솟값', '최댓값', '표준편차']
summary1['감소한 자치구 수'] = [is_decrease(data['ntl_chg']).sum(), is_decrease(data['no2_chg']).sum()]
display(summary1)

# --- 5-4. 자치구별 값과 변화율 표 (NO2 변화율 순으로 정렬)
table1 = data[['자치구', 'y2019_ntl', 'y2020_ntl', 'ntl_chg', 'y2019_no2', 'y2020_no2', 'no2_chg']].copy()
table1.columns = ['자치구', '야간광량 2019', '야간광량 2020', TXT_NTL, 'NO₂ 2019', 'NO₂ 2020', TXT_NO2]
display(table1.sort_values(TXT_NO2, ascending=(CHANGE_DEF != 'diff')).reset_index(drop=True))

# --- 5-5. 변화율 지도: 두 지표를 나란히 그림
# 공간적 분포 지도 (감소 = 파랑, 증가 = 빨강)
gdf = gu.merge(data, on='자치구')
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, col, title in zip(axes, ['ntl_chg', 'no2_chg'], [LBL_NTL, LBL_NO2]):
    # 0을 가운데 색(흰색)으로 맞추기 위해 색 범위를 ±최댓값으로 대칭
    lim = np.nanmax(np.abs(gdf[col]))
    cmap = 'RdBu' if CHANGE_DEF == 'diff' else 'RdBu_r'   # 감소를 항상 파란색으로
    gdf.plot(column=col, cmap=cmap, vmin=-lim, vmax=lim, edgecolor='k', linewidth=0.5, legend=True, ax=ax)
    # 각 자치구 안쪽에 이름과 변화율 숫자 표시
    for _, r in gdf.iterrows():
        p = r.geometry.representative_point()
        ax.annotate(f"{r['자치구']}\n{r[col]:.3g}", (p.x, p.y), ha='center', va='center', fontsize=6.5)
    ax.set_title(title); ax.axis('off')
fig.suptitle('2019년 대비 2020년 3~5월 자치구별 변화 (파랑 = 감소, 빨강 = 증가)')
plt.tight_layout(); plt.savefig(f'{LOCAL_OUT}/fig1_change_maps.png', dpi=200); plt.show()

## 6. 세부목적 (2): 25개 자치구 전체의 회귀분석과 상관관계
- **회귀모형**: NO₂ 변화율 = β₀ + β₁ × 야간광량 변화율 + ε (최소제곱법)
- β₁ 해석: 야간광량 변화율이 1%p 클 때 NO₂ 변화율이 평균적으로 β₁%p 다르게 나타남
- 평가 지표: 기울기 β₁과 95% 신뢰구간, p값, 결정계수 R², Pearson r(=회귀의 상관계수)과 95% 신뢰구간
- 보조: 잔차 정규성(Shapiro-Wilk), 순위 기반 Spearman ρ (이상값에 덜 민감)

In [ ]:
# ▶ [6단계] 세부목적 (2): 25개 자치구 전체 회귀분석(NO2 변화율 ~ 야간광량 변화율)과 상관관계

# --- 6-1. 회귀분석 함수: 한 집단의 자치구들로 "NO2 변화율 = β0 + β1 × 야간광량 변화율"을 추정
def regress(df, label):
    # x = 야간광량 변화율(가로축), y = NO2 변화율(세로축)
    x, y = df['ntl_chg'].astype(float).values, df['no2_chg'].astype(float).values
    n = len(x)
    row = {'집단': label, 'n': n, '자치구': ', '.join(df['자치구'])}
    if n < 3:
        return {**row, '비고': '표본 3개 미만: 계산 불가'}, None
    # 최소제곱법(OLS) 회귀모형 적합. add_constant는 절편 β0를 넣기 위한 것
    model = sm.OLS(y, sm.add_constant(x)).fit()
    # 기울기 β1의 95% 신뢰구간
    lo, hi = model.conf_int(alpha=ALPHA)[1]
    # Pearson r = 기울기 부호 × √R² (단순회귀에서 상관계수와 같음)
    r = np.sign(model.params[1]) * np.sqrt(model.rsquared)
    # r의 95% 신뢰구간: Fisher z 변환 사용 (표본 4개 이상일 때)
    if n > 3:
        z, se = np.arctanh(np.clip(r, -0.9999, 0.9999)), 1 / np.sqrt(n - 3)
        r_lo, r_hi = np.tanh([z - 1.96 * se, z + 1.96 * se])
    else:
        r_lo = r_hi = np.nan
    # 보조 지표: Spearman 순위상관(이상값에 덜 민감), 잔차 정규성 검정(Shapiro-Wilk)
    rho, rho_p = stats.spearmanr(x, y)
    resid_p = stats.shapiro(model.resid).pvalue if n >= 3 else np.nan
    # 결과를 한 줄(딕셔너리)로 정리
    row.update({'기울기 β₁': model.params[1], 'β₁ 95% CI 하한': lo, 'β₁ 95% CI 상한': hi, 'β₁ p값': model.pvalues[1],
                '절편 β₀': model.params[0], 'R²': model.rsquared, 'Pearson r': r, 'r 95% CI': f'[{r_lo:.2f}, {r_hi:.2f}]',
                'Spearman ρ': rho, 'ρ p값': rho_p, '잔차 정규성 p': resid_p,
                '유의(β₁)': '유의' if model.pvalues[1] < ALPHA else '유의하지 않음',
                '비고': '표본이 적어 해석 주의' if n < 8 else ''})
    return row, model

# --- 6-2. 세부목적 (2): 25개 자치구 전체로 회귀분석 실행
overall, overall_model = regress(data, '서울시 전체')
result2 = pd.DataFrame([overall]).set_index('집단').drop(columns='자치구')
display(result2.T)
# statsmodels의 상세 회귀 결과표
print(overall_model.summary())

# ---- 모든 회귀 그래프에 쓸 공통 축 범위 (25개 자치구 전체 자료 기준, 여백 10%) ----
def padded(lo, hi, pad=0.10):
    lo, hi = min(lo, 0), max(hi, 0)          # 0(변화 없음) 기준선이 항상 보이도록
    d = (hi - lo) * pad
    return lo - d, hi + d
XLIM = padded(data['ntl_chg'].min(), data['ntl_chg'].max())
YLIM = padded(data['no2_chg'].min(), data['no2_chg'].max())
XS = np.linspace(*XLIM, 200)
def style_axes(ax):
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_xlabel(LBL_NTL); ax.set_ylabel(LBL_NO2)
print(f'공통 축 범위: 가로(야간광량) {XLIM[0]:.2f} ~ {XLIM[1]:.2f}, 세로(NO2) {YLIM[0]:.2f} ~ {YLIM[1]:.2f}')

# --- 6-3. 그림 2: 전체 회귀 산점도 (점 = 자치구, 검은 선 = 회귀선, 회색 띠 = 95% 신뢰대)
fig, ax = plt.subplots(figsize=(7, 5.8))
ax.scatter(data['ntl_chg'], data['no2_chg'], s=45, color='#2b6cb0', zorder=3)
for _, r in data.iterrows():
    ax.annotate(r['자치구'], (r['ntl_chg'], r['no2_chg']), fontsize=7, xytext=(3, 3), textcoords='offset points')
# 회귀선과 신뢰대를 공통 축 범위 전체에 걸쳐 계산
pred = overall_model.get_prediction(sm.add_constant(XS)).summary_frame(alpha=ALPHA)
ax.plot(XS, pred['mean'], 'k-', lw=1.2, label='회귀선')
ax.fill_between(XS, pred['mean_ci_lower'], pred['mean_ci_upper'], color='gray', alpha=0.2, label='95% 신뢰대')
# 제목에 회귀식, R², r, p값 표시
b0, b1 = overall_model.params
ax.set_title(f"서울시 25개 자치구: y = {b1:.3g}x {'+' if b0 >= 0 else '−'} {abs(b0):.3g}\n"
             f"R² = {overall['R²']:.2f}, r = {overall['Pearson r']:.2f}, p = {overall['β₁ p값']:.3g}")
style_axes(ax); ax.legend(loc='best', fontsize=8)
plt.tight_layout(); plt.savefig(f'{LOCAL_OUT}/fig2_regression_overall.png', dpi=200); plt.show()

## 7. 세부목적 (3)-①: 토지 용도별 대표 집단 확인
1번 셀의 `GROUPS`(입지계수 LQ로 선정한 집단)를 불러와 25개 자치구가 빠짐없이, 겹치지 않게 들어 있는지 확인하고 지도로 보여 줍니다.

In [ ]:
# ▶ [7단계] 세부목적 (3)-①: 토지 용도별 대표 집단 확인과 집단 지도

# --- 7-1. 집단 이름과 색 정하기
ZONES = list(GROUPS)
ZCOLOR = dict(zip(ZONES, ['#1f77b4', '#d62728', '#7f7f7f', '#2ca02c']))   # 상업 파랑, 주거 빨강, 공업 회색, 녹지 초록

# --- 7-2. 점검: 이름 오타, 두 집단에 중복된 구, 빠진 구가 없는지 확인
all_members = [g for v in GROUPS.values() for g in v]
unknown = set(all_members) - set(GU_NAMES)
dups = {g for g in all_members if all_members.count(g) > 1}
missing = set(GU_NAMES) - set(all_members)
assert not unknown, f'GROUPS에 자치구 이름이 잘못된 항목: {unknown}'
if dups:    print('⚠️ 두 집단 이상에 들어간 자치구:', dups)
if missing: print('⚠️ 어느 집단에도 없는 자치구:', missing)

# --- 7-3. 집단 목록 출력, 각 자치구에 소속 집단 표시
groups = GROUPS
for z in ZONES:
    print(f"{z} ({len(groups[z])}개): {', '.join(groups[z])}")
data['집단'] = data['자치구'].map({g: z for z, v in groups.items() for g in v})

# --- 7-4. 그림 3: 집단별 색으로 칠한 서울 지도
fig, ax = plt.subplots(figsize=(7, 5.5))
g = gu.merge(data[['자치구', '집단']], on='자치구')
for z in ZONES:
    g[g['집단'] == z].plot(color=ZCOLOR[z], edgecolor='k', linewidth=0.5, ax=ax, label=z)
for _, r in g.iterrows():
    p = r.geometry.representative_point(); ax.annotate(r['자치구'], (p.x, p.y), ha='center', va='center', fontsize=7)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=ZCOLOR[z], label=f'{z} ({len(groups[z])}개)') for z in ZONES], loc='lower left', fontsize=8)
ax.set_title('토지 용도별 대표 집단'); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{LOCAL_OUT}/fig3_groups_map.png', dpi=200); plt.show()

## 8. 세부목적 (3)-②: 집단별 회귀분석과 서울시 전체와의 비교
- 각 집단 내부에서 6번 셀과 같은 회귀분석(NO₂ 변화율 = β₀ + β₁ × 야간광량 변화율)을 실행하고, 기울기·R²·r을 서울시 전체와 비교합니다.
- 그림 4: 한 그래프에 4개 집단의 회귀선 / 그림 5: 집단별 그래프 4개(회귀선 1개씩)
- 모든 회귀 그래프(그림 2, 4, 5)는 **가로축(야간광량 변화율)과 세로축(NO₂ 변화율)의 범위가 같습니다.**
- ⚠️ 집단별 자치구가 5~8개로 적어 신뢰구간이 넓습니다. **표본 수(n)를 함께 제시하고 결과는 경향 수준으로 해석**하세요.

In [ ]:
# ▶ [8단계] 세부목적 (3)-②: 집단별 회귀분석 → 서울시 전체와 비교 → 그래프 2종

# --- 8-1. 세부목적 (3): 집단마다 6단계와 같은 회귀분석 실행 (첫 줄은 비교 기준인 서울시 전체)
rows, models = [overall], {'서울시 전체': overall_model}
for z in ZONES:
    row, model = regress(data[data['자치구'].isin(groups[z])], z)
    rows.append(row); models[z] = model
result3 = pd.DataFrame(rows).set_index('집단')
# 보고서에 쓸 주요 지표만 골라 표로 보여 줌
cols_show = ['n', '자치구', '기울기 β₁', 'β₁ 95% CI 하한', 'β₁ 95% CI 상한', 'β₁ p값', '절편 β₀', 'R²', 'Pearson r', 'r 95% CI', 'Spearman ρ', 'ρ p값', '유의(β₁)', '비고']
display(result3[[c for c in cols_show if c in result3.columns]])

# --- 8-2. 서울시 전체와 비교: 기울기 차이, 상관계수 차이, 관계 방향
comp = result3[['n', '기울기 β₁', 'R²', 'Pearson r']].copy()
comp['기울기 차이(집단−전체)'] = comp['기울기 β₁'] - overall['기울기 β₁']
comp['r 차이(집단−전체)'] = comp['Pearson r'] - overall['Pearson r']
comp['방향'] = np.where(comp['기울기 β₁'] > 0, '양(+)', '음(−)')
display(comp)

# 범례에 넣을 문구: 집단 이름, 표본 수, 기울기, R²
def group_label(z):
    m = models[z]; n = len(groups[z])
    return f'{z} (n={n}, β$_1$={m.params[1]:.3g}, R²={m.rsquared:.2f})' if m is not None else f'{z} (n={n}, 표본 부족)'

# 그림 4: 한 그래프에 4개 집단의 회귀선
fig, ax = plt.subplots(figsize=(8, 6.2))
# 집단별 점과 이름을 집단 색으로 표시
for z in ZONES:
    sub = data[data['자치구'].isin(groups[z])]
    ax.scatter(sub['ntl_chg'], sub['no2_chg'], s=35, color=ZCOLOR[z], alpha=0.75, zorder=2)
    for _, r in sub.iterrows():
        ax.annotate(r['자치구'], (r['ntl_chg'], r['no2_chg']), fontsize=6.5, color=ZCOLOR[z], xytext=(3, 3), textcoords='offset points')
    m = models[z]
    if m is not None:
        sx = np.linspace(sub['ntl_chg'].min(), sub['ntl_chg'].max(), 50)   # 각 집단 자료 범위 안에서만 선을 그림
        ax.plot(sx, m.params[0] + m.params[1] * sx, color=ZCOLOR[z], lw=2.2, label=group_label(z), zorder=3)
ax.set_title('토지 용도별 대표 집단의 회귀선 비교')
style_axes(ax); ax.legend(loc='best', fontsize=8)
plt.tight_layout(); plt.savefig(f'{LOCAL_OUT}/fig4_group_regressions_combined.png', dpi=200); plt.show()

# 그림 5: 집단별로 따로 (그래프 4개, 회귀선 1개씩)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, z in zip(axes.ravel(), ZONES):
    sub = data[data['자치구'].isin(groups[z])]
    # 회색 점 = 해당 집단이 아닌 자치구 (위치 비교용 배경)
    ax.scatter(data['ntl_chg'], data['no2_chg'], s=20, color='#e5e5e5', zorder=1)       # 나머지 자치구(배경)
    ax.scatter(sub['ntl_chg'], sub['no2_chg'], s=45, color=ZCOLOR[z], zorder=3)
    for _, r in sub.iterrows():
        ax.annotate(r['자치구'], (r['ntl_chg'], r['no2_chg']), fontsize=7, xytext=(3, 3), textcoords='offset points')
    # 해당 집단의 회귀선과 회귀식·R²·p값 제목
    m = models[z]
    if m is not None:
        sx = np.linspace(sub['ntl_chg'].min(), sub['ntl_chg'].max(), 50)
        ax.plot(sx, m.params[0] + m.params[1] * sx, color=ZCOLOR[z], lw=2)
        r_ = result3.loc[z]
        ax.set_title(f"{z} (n={len(sub)})\ny = {m.params[1]:.3g}x {'+' if m.params[0] >= 0 else '−'} {abs(m.params[0]):.3g}, "
                     f"R² = {m.rsquared:.2f}, p = {r_['β₁ p값']:.3g}")
    else:
        ax.set_title(f'{z} (n={len(sub)}): 표본 부족으로 회귀분석 불가')
    # 축 범위를 그림 2·4와 똑같이 맞춤
    style_axes(ax)
plt.tight_layout(); plt.savefig(f'{LOCAL_OUT}/fig5_group_regressions_separate.png', dpi=200); plt.show()

## 9. 결과 저장

In [ ]:
# ▶ [9단계] 결과 저장: 엑셀 파일 + 그림 파일, 코랩에서는 압축 파일로 자동 다운로드

# --- 9-1. 모든 결과표를 시트별로 엑셀 파일 하나에 저장
xlsx = f'{LOCAL_OUT}/results.xlsx'
with pd.ExcelWriter(xlsx) as w:
    data.drop(columns=[]).to_excel(w, sheet_name='(1) 자치구별 값과 집단', index=False)
    table1.to_excel(w, sheet_name='(1) 자치구별 값과 변화', index=False)
    summary1.to_excel(w, sheet_name='(1) 변화 분포')
    result2.T.to_excel(w, sheet_name='(2) 전체 회귀·상관')
    pd.DataFrame({z: pd.Series(m, dtype=object) for z, m in groups.items()}).to_excel(w, sheet_name='(3) 토지 용도별 집단', index=False)
    result3.to_excel(w, sheet_name='(3) 집단별 회귀·상관')
    comp.to_excel(w, sheet_name='(3) 전체와 비교')

# --- 9-2. (선택) 구글 드라이브에도 복사
if SAVE_TO_DRIVE:
    import shutil
    for f in os.listdir(LOCAL_OUT):
        shutil.copy(os.path.join(LOCAL_OUT, f), OUT_DIR)
    print('드라이브에도 저장:', OUT_DIR)

# --- 9-3. 저장된 파일 목록 출력, 코랩이면 results.zip으로 묶어 내 컴퓨터로 다운로드
print('저장된 파일:', sorted(os.listdir(LOCAL_OUT)))
if IN_COLAB:
    shutil_ = __import__('shutil'); shutil_.make_archive('/content/results', 'zip', LOCAL_OUT)
    from google.colab import files; files.download('/content/results.zip')